# Restricted 25-class cross-dataset evaluation

Self-contained evaluation notebook. It selects the 25 shared logits from the original 317-class model output.


In [41]:
# 1. Imports and centralized configuration
from pathlib import Path
import json, os, time
from datetime import datetime
import numpy as np, pandas as pd, torch
import numpy, pandas, matplotlib, seaborn
import torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
ROOT=Path(r'G:\Ragib\Prescription')
PRIMARY_DATASET_DIR=ROOT/'Full_Dataset_Split'/'Train'; EXTERNAL_DATASET_DIR=ROOT/'Cross_Test_Final'; MODEL_ROOT=ROOT/'Medicine codes'/'Models'; ARCHIVE_ROOT=ROOT/'Medicine Paper Details'/'Codes and Models'/'Codes'
OUTPUT_ROOT=Path(r'G:\Ragib\Prescription\CrossDataset_3Seed_RESULTS'); PROTOCOL='restricted'; OUT_DIR=OUTPUT_ROOT/PROTOCOL; os.makedirs(OUT_DIR,exist_ok=True)
SEEDS=[42,123,456]; MODEL_ORDER=['vgg16','vgg19','densenet121','mobilenet_v2','efficientnet_b0','cnn_bilstm','cnn_attention']
TOTAL_MODELS=len(MODEL_ORDER); TOTAL_SEEDS=len(SEEDS); TOTAL_RUNS=TOTAL_MODELS*TOTAL_SEEDS; NUM_CLASSES=317; NUM_EXTERNAL_CLASSES=25
IMAGE_SIZE=(224,224); BATCH_SIZE=16; NUM_WORKERS=0; RUN_EVALUATION=True; DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OVERLAP_CLASSES=['Ace','Axodin','Beklo','Denixil','Disopan','Esonix','Exium','Fenadin','Fexo','Fexofast','Fixal','Flexibac','Lumona','M-kast','Maxima','Maxpro','Monas','Montene','Napa Extend','Odmon','Renova','Rhinil','Sergel','Trilock','Zithrin']
MODEL_GROUP={'vgg16':'baseline','vgg19':'baseline','densenet121':'baseline','mobilenet_v2':'baseline','efficientnet_b0':'baseline','cnn_bilstm':'bilstm','cnn_attention':'attention'}
ARCHIVE_DIRS={'baseline':ARCHIVE_ROOT/'multirun_baseline','bilstm':ARCHIVE_ROOT/'multirun_bilstm','attention':ARCHIVE_ROOT/'multirun_attention'}
CHECKPOINT_FILENAMES={'vgg16':'best_vgg16.pth','vgg19':'best_vgg19.pth','densenet121':'best_densenet121.pth','mobilenet_v2':'best_mobilenetv2.pth','efficientnet_b0':'best_efficientnetb0.pth','cnn_bilstm':'best_bilstm_317.pth','cnn_attention':'best_attention_317.pth'}
RUN_TO_SEED={1:42,2:123,3:456}; SEED_TO_RUN={seed:run for run,seed in RUN_TO_SEED.items()}
RUN_COMPLETE_MODEL_NAMES={'mobilenet_v2':'mobilenetv2','efficientnet_b0':'efficientnetb0'}
CHECKPOINTS={model:{seed:ARCHIVE_DIRS[MODEL_GROUP[model]]/f'run_{run}'/CHECKPOINT_FILENAMES[model] for run,seed in RUN_TO_SEED.items()} for model in MODEL_ORDER}
RUN_COMPLETE_PATHS={model:{seed:ARCHIVE_DIRS[MODEL_GROUP[model]]/f'run_{run}'/'run_complete.json' for run,seed in RUN_TO_SEED.items()} for model in MODEL_ORDER}
IMAGE_TRANSFORM=transforms.Compose([transforms.Resize(IMAGE_SIZE),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
print(f'Configuration ready | protocol={PROTOCOL} | device={DEVICE} | intended runs={TOTAL_RUNS}')


Configuration ready | protocol=restricted | device=cuda | intended runs=21


In [42]:
# 2-4. Environment, paths, datasets, mapping, and DataLoader
required=['os','json','time','numpy','pandas','torch','matplotlib','seaborn','np','pd','plt','sns','OUTPUT_ROOT','PROTOCOL','OUT_DIR','SEEDS','MODEL_ORDER','TOTAL_MODELS','TOTAL_SEEDS','TOTAL_RUNS','DEVICE']
missing=[x for x in required if x not in globals()]
if missing: raise RuntimeError('Missing configuration names: '+', '.join(missing))
if PROTOCOL!='restricted': raise ValueError('Only restricted protocol is supported.')
for label,path in [('primary dataset',PRIMARY_DATASET_DIR),('external dataset',EXTERNAL_DATASET_DIR),('model directory',MODEL_ROOT)]:
    if not path.is_dir(): raise FileNotFoundError(f'{label} does not exist: {path}')
primary_ds=datasets.ImageFolder(PRIMARY_DATASET_DIR); external_ds=datasets.ImageFolder(EXTERNAL_DATASET_DIR,transform=IMAGE_TRANSFORM)
primary_classes=list(primary_ds.classes); external_classes=list(external_ds.classes)
if len(primary_classes)!=NUM_CLASSES: raise ValueError(f'Expected 317 primary classes, found {len(primary_classes)}')
if set(external_classes)!=set(OVERLAP_CLASSES) or len(external_classes)!=25: raise ValueError('External folders do not exactly match the 25 configured classes.')
if not set(external_classes).issubset(primary_classes): raise ValueError('An external class is missing from the primary class mapping.')
counts=np.bincount(np.asarray(external_ds.targets),minlength=25)
if len(external_ds)!=250: raise ValueError(f'Expected 250 external images, found {len(external_ds)}')
if not np.all(counts==10): raise ValueError(f'Expected 10 images per external class, found {counts.tolist()}')
external_global_ids=np.asarray([primary_ds.class_to_idx[x] for x in external_classes],dtype=np.int64)
if len(set(external_global_ids.tolist()))!=25: raise ValueError('Class mapping is not one-to-one.')
loader=DataLoader(external_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')
if not loader: raise RuntimeError('DataLoader could not be created.')
print(f'Validated primary={len(primary_classes)} classes | external={len(external_classes)} classes | images={len(external_ds)} | batches={len(loader)} | device={DEVICE}')


Validated primary=317 classes | external=25 classes | images=250 | batches=16 | device=cuda


In [43]:
# 5. Model architecture definitions
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__(); self.mobilenet=models.mobilenet_v2(weights=None); self.mobilenet.classifier=nn.Identity(); self.efficientnet=models.efficientnet_b0(weights=None); self.efficientnet.classifier=nn.Identity(); self.lstm=nn.LSTM(2560,256,batch_first=True,bidirectional=True); self.classifier=nn.Sequential(nn.Linear(512,256),nn.ReLU(),nn.Dropout(.5),nn.Linear(256,NUM_CLASSES))
    def forward(self,x):
        z=torch.cat([self.mobilenet(x),self.efficientnet(x)],1).unsqueeze(1); z,_=self.lstm(z); return self.classifier(z[:,-1])
class Attention(nn.Module):
    def __init__(self):
        super().__init__(); self.attn=nn.Sequential(nn.Linear(2560,1280),nn.ReLU(),nn.Linear(1280,2),nn.Softmax(1))
    def forward(self,a,b):
        w=self.attn(torch.cat([a,b],1)); return w[:,:1]*a+w[:,1:2]*b
class CNNAttention(nn.Module):
    def __init__(self):
        super().__init__(); self.mobilenet=models.mobilenet_v2(weights=None); self.mobilenet.classifier=nn.Identity(); self.efficientnet=models.efficientnet_b0(weights=None); self.efficientnet.classifier=nn.Identity(); self.fusion=Attention(); self.classifier=nn.Sequential(nn.Linear(1280,256),nn.ReLU(),nn.Dropout(.5),nn.Linear(256,NUM_CLASSES))
    def forward(self,x): return self.classifier(self.fusion(self.mobilenet(x),self.efficientnet(x)))
def build_model(name):
    if name=='vgg16': m=models.vgg16(weights=None); m.classifier[6]=nn.Linear(4096,NUM_CLASSES)
    elif name=='vgg19': m=models.vgg19(weights=None); m.classifier[6]=nn.Linear(4096,NUM_CLASSES)
    elif name=='densenet121': m=models.densenet121(weights=None); m.classifier=nn.Linear(m.classifier.in_features,NUM_CLASSES)
    elif name=='mobilenet_v2': m=models.mobilenet_v2(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
    elif name=='efficientnet_b0': m=models.efficientnet_b0(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,NUM_CLASSES)
    elif name=='cnn_bilstm': m=BiLSTM()
    elif name=='cnn_attention': m=CNNAttention()
    else: raise ValueError(f'Unknown model: {name}')
    return m
def unwrap_state_dict(x):
    if isinstance(x,dict):
        for k in ('model','model_state','model_state_dict','state_dict'):
            if isinstance(x.get(k),dict): return x[k]
        if x and all(isinstance(v,torch.Tensor) for v in x.values()): return x
    raise TypeError('Unsupported checkpoint wrapper.')
def normalized_state_dict(path):
    raw=torch.load(path,map_location='cpu',weights_only=False); return {k.removeprefix('module.'):v for k,v in unwrap_state_dict(raw).items()}


In [44]:
# 6. Checkpoint configuration/loading and full preflight
def load_model(name,path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Checkpoint does not exist: {path}')
    model=build_model(name); missing,unexpected=model.load_state_dict(normalized_state_dict(path),strict=False)
    if missing or unexpected: raise RuntimeError(f'Checkpoint key mismatch for {name}: missing={missing}, unexpected={unexpected}')
    return model.to(DEVICE).eval()
def read_run_record(path,name):
    data=json.loads(path.read_text(encoding='utf-8'))
    records=data if isinstance(data,list) else [data]
    matches=[record for record in records if not isinstance(data,list) or record.get('Model',name)==name]
    if len(matches)!=1: raise RuntimeError(f'Expected exactly one run record for {name} in {path}, found {len(matches)}')
    return matches[0]
available_runs=[]
for name in MODEL_ORDER:
    for seed in SEEDS:
        path=CHECKPOINTS[name][seed]; complete=RUN_COMPLETE_PATHS[name][seed]
        if not path.is_file(): raise FileNotFoundError(f'MISSING checkpoint for {name}, seed {seed}: {path}')
        if not complete.is_file(): raise FileNotFoundError(f'MISSING run_complete.json for {name}, seed {seed}: {complete}')
        record=read_run_record(complete,RUN_COMPLETE_MODEL_NAMES.get(name,name)); recorded_seed=record.get('Random Seed')
        if recorded_seed!=seed: raise RuntimeError(f'SEED MISMATCH before evaluation: {name}, expected {seed}, recorded {recorded_seed}, metadata={complete}')
        probe=load_model(name,path); del probe
        if DEVICE.type=='cuda': torch.cuda.empty_cache()
        run_number=SEED_TO_RUN[seed]
        available_runs.append({'model':name,'seed':seed,'seed_status':'verified','run_number':run_number,'checkpoint':str(path),'run_complete':str(complete),'record':record})
        print(f'VALIDATED model={name} | run_{run_number} | seed={seed} | checkpoint={path}',flush=True)
if len(available_runs)!=TOTAL_RUNS: raise RuntimeError(f'Expected {TOTAL_RUNS} validated runs, found {len(available_runs)}')
print(f'All {len(available_runs)}/{TOTAL_RUNS} checkpoint paths and recorded seeds validated before evaluation.')


VALIDATED model=vgg16 | run_1 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg16.pth
VALIDATED model=vgg16 | run_2 | seed=123 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_2\best_vgg16.pth
VALIDATED model=vgg16 | run_3 | seed=456 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_3\best_vgg16.pth
VALIDATED model=vgg19 | run_1 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg19.pth
VALIDATED model=vgg19 | run_2 | seed=123 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_2\best_vgg19.pth
VALIDATED model=vgg19 | run_3 | seed=456 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_3\best_vgg19.pth
VALIDATED model=densenet121 | run_1 

In [45]:
# 7. Evaluation and metric/confusion-matrix functions
def evaluate_model(model,run):
    yt=[]; yp=[]; ypg=[]; top=[]; started=time.perf_counter()
    print(f'Evaluation protocol={PROTOCOL.upper()} | device={DEVICE} | model={run.get("model")} | seed={run.get("seed_status")}')
    with torch.inference_mode():
        for i,(images,labels) in enumerate(loader,1):
            logits=model(images.to(DEVICE));
            if logits.ndim!=2 or logits.shape[1]!=NUM_CLASSES: raise ValueError(f'Expected 317 logits, got {tuple(logits.shape)}')
            selected=logits.index_select(1,torch.as_tensor(external_global_ids,device=DEVICE)); p=selected.argmax(1).cpu().numpy(); t=torch.topk(selected,5,1).indices.cpu().numpy()
            yt.extend(labels.numpy()); yp.extend(p); ypg.extend(external_global_ids[p]); top.extend(t); print(f'Batch {i}/{len(loader)} | elapsed={time.perf_counter()-started:.1f}s',flush=True)
    yt,yp,ypg=map(np.asarray,(yt,yp,ypg)); labels25=np.arange(25); ma=precision_recall_fscore_support(yt,yp,labels=labels25,average='macro',zero_division=0); we=precision_recall_fscore_support(yt,yp,labels=labels25,average='weighted',zero_division=0)
    return {'y_true':yt,'y_pred':yp,'y_pred_global':ypg,'top5':top,'top1':float(accuracy_score(yt,yp)),'top5_accuracy':float(np.mean([int(a in b) for a,b in zip(yt,top)])),'macro_precision':float(ma[0]),'macro_recall':float(ma[1]),'macro_f1':float(ma[2]),'weighted_precision':float(we[0]),'weighted_recall':float(we[1]),'weighted_f1':float(we[2]),'report':classification_report(yt,yp,labels=labels25,target_names=external_classes,output_dict=True,zero_division=0),'cm':confusion_matrix(yt,yp,labels=labels25),'duration_seconds':time.perf_counter()-started}
def save_cm(cm,title,path):
    plt.figure(figsize=(16,14)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=external_classes,yticklabels=external_classes); plt.title(title); plt.xlabel('Predicted restricted class'); plt.ylabel('True class'); plt.xticks(rotation=90); plt.tight_layout(); plt.savefig(path,dpi=200,bbox_inches='tight'); plt.close()


In [46]:
# 8. Main evaluation loop with progressive saving
metrics=['top1','top5_accuracy','macro_precision','macro_recall','macro_f1','weighted_precision','weighted_recall','weighted_f1']; rows=[]; matrices={}
if not RUN_EVALUATION: print('RUN_EVALUATION=False: preflight complete; no inference started.')
else:
    started=time.perf_counter(); print(f'Total configured models={TOTAL_MODELS} | seeds={TOTAL_SEEDS} | runs={TOTAL_RUNS} | runnable={len(available_runs)}')
    for n,run in enumerate(available_runs,1):
        name=run['model']; matrices.setdefault(name,[]); model=None
        try:
            print(f'[{n}/{len(available_runs)}] Current model={name} | seed={run.get("seed")} | checkpoint={run.get("checkpoint")}',flush=True); model=load_model(name,run['checkpoint']); r=evaluate_model(model,run); d=OUT_DIR/name/f'seed_{run.get("seed")}'; d.mkdir(parents=True,exist_ok=True)
            pd.DataFrame({'true_class':[external_classes[i] for i in r['y_true']],'true_external_index':r['y_true'],'true_primary_index':external_global_ids[r['y_true']],'predicted_class':[external_classes[i] for i in r['y_pred']],'predicted_external_index':r['y_pred'],'predicted_primary_index':r['y_pred_global'],'top5_classes':['|'.join(external_classes[i] for i in x) for x in r['top5']]}).to_csv(d/'predictions.csv',index=False); (d/'classification_report.json').write_text(json.dumps(r['report'],indent=2),encoding='utf-8'); pd.DataFrame(r['report']).T.to_csv(d/'classification_report.csv',index_label='class'); pd.DataFrame(r['cm'],index=external_classes,columns=external_classes).to_csv(d/'confusion_matrix.csv'); save_cm(r['cm'],f'{name.upper()} restricted 25-class | seed {run.get("seed")}',d/'confusion_matrix.png')
            matrices[name].append(r['cm']); rows.append({'model':name,'seed':run['seed'],'run_number':run['run_number'],'checkpoint':run['checkpoint'],'n_test':len(r['y_true']),**{k:r[k] for k in metrics}}); pd.DataFrame(rows).to_csv(OUT_DIR/'completed_runs_progressive.csv',index=False); print(f'Model-level completion: {name} seed={run.get("seed")}',flush=True)
        except Exception as exc: print(f'RUN FAILED | model={name} | seed={run.get("seed")} | checkpoint={run.get("checkpoint")} | {type(exc).__name__}: {exc}',flush=True)
        finally:
            if model is not None: del model
            if DEVICE.type=='cuda': torch.cuda.empty_cache()
    print(f'Overall completion: {len(rows)}/{len(available_runs)} in {(time.perf_counter()-started)/60:.2f} minutes')


Total configured models=7 | seeds=3 | runs=21 | runnable=21
[1/21] Current model=vgg16 | seed=42 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_1\best_vgg16.pth
Evaluation protocol=RESTRICTED | device=cuda | model=vgg16 | seed=verified
Batch 1/16 | elapsed=0.2s
Batch 2/16 | elapsed=0.3s
Batch 3/16 | elapsed=0.4s
Batch 4/16 | elapsed=0.5s
Batch 5/16 | elapsed=0.5s
Batch 6/16 | elapsed=0.6s
Batch 7/16 | elapsed=0.7s
Batch 8/16 | elapsed=0.8s
Batch 9/16 | elapsed=0.8s
Batch 10/16 | elapsed=0.9s
Batch 11/16 | elapsed=1.0s
Batch 12/16 | elapsed=1.1s
Batch 13/16 | elapsed=1.1s
Batch 14/16 | elapsed=1.2s
Batch 15/16 | elapsed=1.3s
Batch 16/16 | elapsed=1.3s
Model-level completion: vgg16 seed=42
[2/21] Current model=vgg16 | seed=123 | checkpoint=G:\Ragib\Prescription\Medicine Paper Details\Codes and Models\Codes\multirun_baseline\run_2\best_vgg16.pth
Evaluation protocol=RESTRICTED | device=cuda | model=vgg16 | seed=verified
Batch 1/16 | e

In [47]:
# 9. Final result export and summary
if RUN_EVALUATION and rows:
    per_run=pd.DataFrame(rows); per_run.to_csv(OUT_DIR/'all_available_runs_results.csv',index=False); summary=[]
    for name in MODEL_ORDER:
        q=per_run[per_run.model==name]; row={'model':name,'available_runs':len(q),'seed_status':'verified_42_123_456'}
        for metric in metrics:
            if len(q): row[metric+'_mean']=float(q[metric].mean()); row[metric+'_sample_sd']=float(q[metric].std(ddof=1)) if len(q)>1 else float('nan')
        summary.append(row)
    pd.DataFrame(summary).to_csv(OUT_DIR/'summary_available_runs.csv',index=False)
    for name,arr in matrices.items():
        a=np.stack(arr); d=OUT_DIR/name; d.mkdir(parents=True,exist_ok=True); pd.DataFrame(a.sum(0),index=external_classes,columns=external_classes).to_csv(d/'confusion_matrix_sum_available_runs.csv'); pd.DataFrame(a.mean(0),index=external_classes,columns=external_classes).to_csv(d/'confusion_matrix_mean_available_runs.csv')
    print('Results saved under',OUT_DIR)
else: print('No inference results to export in this run.')
print(f'Final status: {len(available_runs)}/{TOTAL_RUNS} verified seed-specific runs completed.')
print('Notebook preflight and dependency audit completed successfully.')


Results saved under G:\Ragib\Prescription\CrossDataset_3Seed_RESULTS\restricted
Final status: 21/21 verified seed-specific runs completed.
Notebook preflight and dependency audit completed successfully.
